# late_fusion_agent — Colab Notebook

End-to-end pipeline: Model A (`PriceTransformer` on candle tokens) and
Model B (`IndicatorModel` on indicator tokens) are trained independently,
then a tiny **MetaModel** learns to combine their val-set logits.

Self-contained: all source inlined from `token_first_transformer`,
`indicator_tokenizer`, and `late_fusion_agent`.

Recommended runtime: **T4 GPU**. Run all cells top-to-bottom.


In [ ]:
!pip install -q pyarrow polars pyyaml scikit-learn pandas
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())


## 1. Tokenizers (delta + bucket)


In [ ]:
"""Shared tokenizers (copied from token_first_transformer/tokenizer/)."""
from __future__ import annotations

from pathlib import Path

import numpy as np


class DeltaTokenizer:
    """Percentage-delta tokenizer (0=PAD, 1=CLS, 2+=bins)."""

    def __init__(self, range_pct: float = 3.0, step_pct: float = 0.05) -> None:
        self.range_pct = range_pct
        self.step_pct = step_pct
        self.n_bins = int(2 * range_pct / step_pct)
        self.pad_id = 0
        self.cls_id = 1
        self._offset = 2
        self.vocab_size = self._offset + self.n_bins

    def encode_batch(self, deltas: np.ndarray) -> np.ndarray:
        deltas_pct = deltas * 100
        bin_idx = np.round(deltas_pct / self.step_pct).astype(np.int32) + self.n_bins // 2 - 1
        bin_idx = np.clip(bin_idx, 0, self.n_bins - 1)
        return (bin_idx + self._offset).astype(np.int32)

    def from_closes(self, closes: np.ndarray) -> np.ndarray:
        n = len(closes)
        ids = np.full(n, self.pad_id, dtype=np.int32)
        if n < 2:
            return ids
        deltas = (closes[1:] - closes[:-1]) / closes[:-1]
        ids[1:] = self.encode_batch(deltas)
        return ids


class BucketTokenizer:
    """Quantile bucket tokenizer (0=PAD, 1=reserved CLS, 2+=bins)."""

    def __init__(self, n_bins: int = 8) -> None:
        self.n_bins = n_bins
        self.pad_id = 0
        self._offset = 2
        self.vocab_size = self._offset + n_bins
        self.boundaries = None

    def fit(self, values: np.ndarray) -> None:
        quantiles = np.linspace(0, 100, self.n_bins + 1)[1:-1]
        self.boundaries = np.percentile(values, quantiles).astype(np.float32)

    def encode_batch(self, values: np.ndarray) -> np.ndarray:
        assert self.boundaries is not None
        bin_idx = np.searchsorted(self.boundaries, values, side="right")
        return (bin_idx + self._offset).astype(np.int32)


## 2. Indicator computer


In [ ]:
"""Technical-indicator computer (from indicator_tokenizer/indicators/computer.py)."""
import numpy as np


def _ema(arr, span):
    alpha = 2.0 / (span + 1)
    out = np.zeros(len(arr), dtype=np.float64)
    out[0] = arr[0]
    for i in range(1, len(arr)):
        out[i] = alpha * arr[i] + (1 - alpha) * out[i - 1]
    return out


class IndicatorComputer:
    def rsi(self, close, period=14):
        delta = np.diff(close.astype(np.float64), prepend=close[0])
        gain = np.where(delta > 0, delta, 0.0)
        loss = np.where(delta < 0, -delta, 0.0)
        avg_gain = _ema(gain, period)
        avg_loss = _ema(loss, period)
        rs = avg_gain / (avg_loss + 1e-10)
        return (100 - 100 / (1 + rs)).astype(np.float32)

    def macd_hist(self, close, fast=12, slow=26, signal=9):
        c = close.astype(np.float64)
        macd_line = _ema(c, fast) - _ema(c, slow)
        signal_line = _ema(macd_line, signal)
        return (macd_line - signal_line).astype(np.float32)

    def bollinger_pctb(self, close, period=20, num_std=2.0):
        c = close.astype(np.float64)
        out = np.zeros(len(c), dtype=np.float32)
        for i in range(period - 1, len(c)):
            w = c[i - period + 1 : i + 1]
            sma = w.mean(); std = w.std()
            upper = sma + num_std * std; lower = sma - num_std * std
            out[i] = (c[i] - lower) / (upper - lower + 1e-10)
        return out

    def atr(self, high, low, close, period=14):
        tr = np.zeros(len(close), dtype=np.float64)
        tr[0] = high[0] - low[0]
        tr[1:] = np.maximum(
            high[1:] - low[1:],
            np.maximum(
                np.abs(high[1:].astype(np.float64) - close[:-1].astype(np.float64)),
                np.abs(low[1:].astype(np.float64) - close[:-1].astype(np.float64)),
            ),
        )
        return _ema(tr, period).astype(np.float32)

    def volume_ratio(self, close, volume, period=20):
        v = volume.astype(np.float64)
        out = np.zeros(len(v), dtype=np.float32)
        for i in range(period - 1, len(v)):
            sma = v[i - period + 1 : i + 1].mean()
            out[i] = v[i] / (sma + 1e-10)
        return out

    def price_vs_sma(self, close, period=20):
        c = close.astype(np.float64)
        out = np.zeros(len(c), dtype=np.float32)
        for i in range(period - 1, len(c)):
            sma = c[i - period + 1 : i + 1].mean()
            out[i] = (c[i] - sma) / (sma + 1e-10)
        return out

    def compute_all(self, ohlcv):
        return {
            "rsi": self.rsi(ohlcv["close"]),
            "macd_hist": self.macd_hist(ohlcv["close"]),
            "bollinger_pctb": self.bollinger_pctb(ohlcv["close"]),
            "atr": self.atr(ohlcv["high"], ohlcv["low"], ohlcv["close"]),
            "volume_ratio": self.volume_ratio(ohlcv["close"], ohlcv["volume"]),
            "price_vs_sma": self.price_vs_sma(ohlcv["close"]),
        }


## 3. Indicator tokenizer


In [ ]:
"""Indicator tokenizer (from indicator_tokenizer/indicators/tokenizer.py)."""
from pathlib import Path
import numpy as np


class FixedBoundaries:
    def __init__(self, bins, offset=2):
        self.bins = np.array(bins, dtype=np.float32)
        self.offset = offset
        self.vocab_size = len(bins) + 1 + offset

    def encode_batch(self, values):
        return (np.searchsorted(self.bins, values, side="right") + self.offset).astype(np.int32)

    def save(self, path): np.save(path, self.bins)
    def load(self, path): self.bins = np.load(path)


class QuantileBoundaries:
    def __init__(self, n_bins, offset=2):
        self.n_bins = n_bins
        self.offset = offset
        self.vocab_size = n_bins + offset
        self.boundaries = None

    def fit(self, values):
        q = np.linspace(0, 100, self.n_bins + 1)[1:-1]
        self.boundaries = np.percentile(values, q).astype(np.float32)

    def encode_batch(self, values):
        assert self.boundaries is not None
        return (np.searchsorted(self.boundaries, values, side="right") + self.offset).astype(np.int32)

    def save(self, path): np.save(path, self.boundaries)
    def load(self, path): self.boundaries = np.load(path)


class IndicatorTokenizer:
    PAD_ID = 0; SPECIAL_ID = 1

    def __init__(self):
        self.rsi = FixedBoundaries(bins=[20, 30, 70, 80])
        self.macd_hist = QuantileBoundaries(n_bins=7)
        self.bollinger_pctb = FixedBoundaries(bins=[0.0, 0.25, 0.75, 1.0])
        self.atr = QuantileBoundaries(n_bins=6)
        self.volume_ratio = QuantileBoundaries(n_bins=5)
        self.price_vs_sma = QuantileBoundaries(n_bins=5)
        self._quantile_fields = ["macd_hist", "atr", "volume_ratio", "price_vs_sma"]
        self._all_fields = ["rsi", "macd_hist", "bollinger_pctb", "atr",
                            "volume_ratio", "price_vs_sma"]

    def fit(self, ind):
        for f in self._quantile_fields:
            getattr(self, f).fit(ind[f])

    def encode(self, ind):
        return {f: getattr(self, f).encode_batch(ind[f]) for f in self._all_fields}

    def vocab_sizes(self):
        return {f: getattr(self, f).vocab_size for f in self._all_fields}

    def save(self, d):
        d = Path(d); d.mkdir(parents=True, exist_ok=True)
        for f in self._all_fields:
            getattr(self, f).save(d / f"{f}.npy")

    def load(self, d):
        d = Path(d)
        for f in self._all_fields:
            getattr(self, f).load(d / f"{f}.npy")


## 4. Model A — PriceTransformer


In [ ]:
"""Model A: PriceTransformer (from token_first_transformer/models/)."""
import torch
import torch.nn as nn


class PriceTransformer(nn.Module):
    def __init__(self, delta_vocab_size=122, bucket_vocab_size=10,
                 delta_emb_dim=64, bucket_emb_dim=16, hidden_dim=256,
                 num_layers=4, num_heads=8, ffn_dim=1024, dropout=0.1,
                 num_classes=3, seq_len=128):
        super().__init__()
        self.delta_emb = nn.Embedding(delta_vocab_size, delta_emb_dim)
        self.vol_emb = nn.Embedding(bucket_vocab_size, bucket_emb_dim)
        self.vb_emb = nn.Embedding(bucket_vocab_size, bucket_emb_dim)
        concat_dim = delta_emb_dim + bucket_emb_dim * 2
        self.proj = nn.Linear(concat_dim, hidden_dim)
        self.pos_emb = nn.Embedding(seq_len, hidden_dim)
        enc = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=ffn_dim, dropout=dropout, activation="gelu",
            batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=num_layers)
        self.norm = nn.LayerNorm(hidden_dim)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim // 2, num_classes))

    def forward(self, delta_ids, vol_ids, vb_ids):
        B, T = delta_ids.shape
        x = torch.cat([self.delta_emb(delta_ids), self.vol_emb(vol_ids),
                       self.vb_emb(vb_ids)], dim=-1)
        x = self.proj(x)
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x = x + self.pos_emb(pos)
        x = self.norm(self.transformer(x))
        return self.head(x[:, 0])


## 5. Model B — IndicatorModel


In [ ]:
"""Model B: IndicatorModel (late_fusion_agent/models/indicator_model.py)."""
import torch
import torch.nn as nn


class IndicatorModel(nn.Module):
    def __init__(self, vocab_sizes, emb_dim=16, hidden_dim=128, num_layers=2,
                 num_heads=4, ffn_dim=256, dropout=0.1, num_classes=3, seq_len=128):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(vs, emb_dim) for vs in vocab_sizes])
        concat_dim = emb_dim * len(vocab_sizes)
        self.proj = nn.Linear(concat_dim, hidden_dim)
        self.pos_emb = nn.Embedding(seq_len + 1, hidden_dim)
        self.cls_token = nn.Parameter(torch.randn(1, 1, hidden_dim) * 0.02)
        enc = nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=num_heads,
            dim_feedforward=ffn_dim, dropout=dropout, activation="gelu",
            batch_first=True)
        self.transformer = nn.TransformerEncoder(enc, num_layers=num_layers)
        self.norm = nn.LayerNorm(hidden_dim)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim // 2, num_classes))

    def forward(self, inputs):
        B = inputs[0].shape[0]
        x = torch.cat([emb(tok) for emb, tok in zip(self.embeddings, inputs)], dim=-1)
        x = self.proj(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        T = x.shape[1]
        pos = torch.arange(T, device=x.device).unsqueeze(0).expand(B, -1)
        x = x + self.pos_emb(pos)
        x = self.norm(self.transformer(x))
        return self.head(x[:, 0])


## 6. MetaModel


In [ ]:
"""Meta-model: small MLP on [logits_a || logits_b] -> 3 classes."""
import torch
import torch.nn as nn


class MetaModel(nn.Module):
    def __init__(self, input_dim=6, hidden_dim=16, num_classes=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, la, lb):
        return self.net(torch.cat([la, lb], dim=-1))


## 7. FusionDataset + collate


In [ ]:
"""FusionDataset: candle tokens + indicator tokens + 3-class label."""
from pathlib import Path
import numpy as np
import pyarrow.parquet as pq


def _load(p):
    t = pq.read_table(p)
    return {c: np.array([v.as_py() for v in t.column(c)], dtype=np.float32) for c in t.column_names}


def _fit_all(files, range_pct, step_pct, n_bins):
    dt = DeltaTokenizer(range_pct=range_pct, step_pct=step_pct)
    rp, lv = [], []
    for f in files:
        d = _load(f)
        if len(d["close"]) < 2: continue
        rp.append((d["high"][1:] - d["low"][1:]) / d["close"][1:])
        lv.append(np.log1p(d["volume"][1:]))
    vt = BucketTokenizer(n_bins=n_bins); vt.fit(np.concatenate(rp))
    bt = BucketTokenizer(n_bins=n_bins); bt.fit(np.concatenate(lv))
    comp = IndicatorComputer()
    keys = ["rsi","macd_hist","bollinger_pctb","atr","volume_ratio","price_vs_sma"]
    ai = {k: [] for k in keys}
    for f in files:
        d = _load(f)
        ind = comp.compute_all({k2: d[k2] for k2 in ["open","high","low","close","volume"]})
        for k in keys: ai[k].append(ind[k])
    it = IndicatorTokenizer(); it.fit({k: np.concatenate(v) for k, v in ai.items()})
    return dt, vt, bt, it, comp


class FusionDataset:
    def __init__(self, files, seq_len=128, target_horizon=60, target_threshold=0.0015,
                 range_pct=3.0, step_pct=0.05, n_bins=8):
        self.seq_len = seq_len
        self.target_horizon = target_horizon
        self.target_threshold = target_threshold
        self.dt, self.vt, self.bt, self.it, self.comp = _fit_all(files, range_pct, step_pct, n_bins)
        frames = [_load(f) for f in files]
        self.closes = np.concatenate([f["close"] for f in frames]).astype(np.float32)
        self.highs = np.concatenate([f["high"] for f in frames]).astype(np.float32)
        self.lows = np.concatenate([f["low"] for f in frames]).astype(np.float32)
        self.volumes = np.concatenate([f["volume"] for f in frames]).astype(np.float32)
        self.opens = np.concatenate([f.get("open", f["close"]) for f in frames]).astype(np.float32)
        self._len = max(0, len(self.closes) - seq_len - target_horizon)

    def __len__(self): return self._len

    def __getitem__(self, idx):
        s, e = idx, idx + self.seq_len
        c, h, l, v, o = (self.closes[s:e], self.highs[s:e], self.lows[s:e],
                         self.volumes[s:e], self.opens[s:e])
        delta = self.dt.from_closes(c); delta[0] = self.dt.cls_id
        rp = np.zeros(self.seq_len, dtype=np.float32)
        rp[1:] = (h[1:] - l[1:]) / c[1:]
        vol = self.vt.encode_batch(rp); vol[0] = self.vt.pad_id
        vb = self.bt.encode_batch(np.log1p(v)); vb[0] = self.bt.pad_id
        ind = self.it.encode(self.comp.compute_all(
            {"open": o, "high": h, "low": l, "close": c, "volume": v}))
        tc = self.closes[e + self.target_horizon - 1]
        cc = self.closes[e - 1]
        d = (tc - cc) / cc
        label = 2 if d > self.target_threshold else (0 if d < -self.target_threshold else 1)
        return delta, vol, vb, ind, label


def collate(batch):
    import torch
    delta = torch.stack([torch.tensor(b[0]) for b in batch]).long()
    vol = torch.stack([torch.tensor(b[1]) for b in batch]).long()
    vb = torch.stack([torch.tensor(b[2]) for b in batch]).long()
    keys = list(batch[0][3].keys())
    ind = [torch.stack([torch.tensor(b[3][k]) for b in batch]).long() for k in keys]
    y = torch.tensor([b[4] for b in batch]).long()
    return delta, vol, vb, ind, y


## 8. FusionTrainer


In [ ]:
"""FusionTrainer: trains Model A and Model B independently, then a MetaModel."""
from pathlib import Path
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from torch.optim import AdamW


class FusionTrainer:
    def __init__(self, model_a, model_b, train_loader, val_loader,
                 epochs_a=3, epochs_b=3, epochs_meta=5,
                 lr=3e-4, weight_decay=0.01, early_stop_patience=3,
                 device="auto", checkpoint_dir=Path("/content/checkpoints")):
        self.device = self._auto_device() if device == "auto" else device
        self.model_a = model_a.to(self.device)
        self.model_b = model_b.to(self.device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.epochs_a = epochs_a; self.epochs_b = epochs_b; self.epochs_meta = epochs_meta
        self.lr = lr; self.weight_decay = weight_decay
        self.early_stop_patience = early_stop_patience
        self.ckpt = Path(checkpoint_dir); self.ckpt.mkdir(parents=True, exist_ok=True)

    @staticmethod
    def _auto_device():
        if torch.cuda.is_available(): return "cuda"
        if torch.backends.mps.is_available(): return "mps"
        return "cpu"

    def train_all(self):
        print("=== Train Model A ===")
        self._train_base(self.model_a, self.epochs_a, "model_a",
            lambda b: self.model_a(b[0].to(self.device), b[1].to(self.device), b[2].to(self.device)))
        print("\n=== Train Model B ===")
        self._train_base(self.model_b, self.epochs_b, "model_b",
            lambda b: self.model_b([t.to(self.device) for t in b[3]]))
        print("\n=== Collect val logits ===")
        la, lb, y = self._collect_logits()
        print("la:", la.shape, "lb:", lb.shape, "y:", y.shape)
        print("\n=== Train Meta-Model ===")
        meta = MetaModel().to(self.device)
        self._train_meta(meta, la, lb, y)
        torch.save(meta.state_dict(), self.ckpt / "meta_model.pt")
        return meta

    def _train_base(self, model, epochs, name, forward_fn):
        opt = AdamW(model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        crit = nn.CrossEntropyLoss()
        best, pat = float("inf"), 0
        for ep in range(1, epochs + 1):
            model.train(mode=True)
            tl, n = 0.0, 0
            for b in self.train_loader:
                loss = crit(forward_fn(b), b[4].to(self.device))
                loss.backward(); opt.step(); opt.zero_grad()
                tl += loss.item(); n += 1
            model.train(mode=False)
            vl, vn = 0.0, 0
            with torch.no_grad():
                for b in self.val_loader:
                    vl += crit(forward_fn(b), b[4].to(self.device)).item(); vn += 1
            vl /= max(vn, 1)
            print(f"  {name} ep{ep}: train={tl/max(n,1):.4f} val={vl:.4f}")
            if vl < best:
                best, pat = vl, 0
                torch.save(model.state_dict(), self.ckpt / f"{name}_best.pt")
            else:
                pat += 1
            if pat >= self.early_stop_patience:
                print(f"  early stop {name} ep{ep}"); break

    def _collect_logits(self):
        self.model_a.train(mode=False); self.model_b.train(mode=False)
        la, lb, y = [], [], []
        with torch.no_grad():
            for b in self.val_loader:
                la.append(self.model_a(b[0].to(self.device), b[1].to(self.device), b[2].to(self.device)).cpu())
                lb.append(self.model_b([t.to(self.device) for t in b[3]]).cpu())
                y.append(b[4])
        return torch.cat(la), torch.cat(lb), torch.cat(y)

    def _train_meta(self, meta, la, lb, y):
        opt = AdamW(meta.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        crit = nn.CrossEntropyLoss()
        ds = TensorDataset(la, lb, y)
        dl = DataLoader(ds, batch_size=64, shuffle=True)
        for ep in range(1, self.epochs_meta + 1):
            meta.train(mode=True)
            tl = 0.0
            for a, b, lbl in dl:
                loss = crit(meta(a.to(self.device), b.to(self.device)), lbl.to(self.device))
                loss.backward(); opt.step(); opt.zero_grad()
                tl += loss.item()
            print(f"  meta ep{ep}: loss={tl/len(dl):.4f}")


## 9. Configuration + data paths


In [ ]:
"""Configuration."""
from pathlib import Path

USE_MOCK_DATA = True
if USE_MOCK_DATA:
    DATA_DIR = Path("/content/mock_data")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_DIR = Path("/content/drive/MyDrive/trading_data")

MOCK_MONTHS = ["2024-01", "2024-02", "2024-03", "2024-04"]
MOCK_MINUTES_PER_MONTH = 20000
SYMBOL = "BTCUSDT"

CFG = {
    "data": {
        "train_months": ["2024-01", "2024-02"],
        "val_months":   ["2024-03", "2024-03"],
        "test_months":  ["2024-04", "2024-04"],
    },
    "sequence": {"length": 128, "target_horizon": 60, "target_threshold": 0.0015},
    "tokenizer": {"delta": {"range_pct": 3.0, "step_pct": 0.05}, "bucket": {"n_bins": 8}},
    "model_a": {"delta_vocab_size": 122, "bucket_vocab_size": 10,
                "delta_emb_dim": 64, "bucket_emb_dim": 16, "hidden_dim": 256,
                "num_layers": 4, "num_heads": 8, "ffn_dim": 1024, "dropout": 0.1,
                "num_classes": 3},
    "model_b": {"vocab_sizes": [7, 9, 7, 8, 7, 7], "emb_dim": 16, "hidden_dim": 128,
                "num_layers": 2, "num_heads": 4, "ffn_dim": 256, "dropout": 0.1,
                "num_classes": 3},
    "training": {"batch_size": 64, "learning_rate": 3e-4, "weight_decay": 0.01,
                 "epochs_a": 3, "epochs_b": 3, "epochs_meta": 5,
                 "early_stop_patience": 3, "device": "auto",
                 "checkpoint_dir": "/content/checkpoints"},
}

if Path("/content/drive/MyDrive").exists():
    ARTIFACTS_ROOT = Path("/content/drive/MyDrive/w_training/late_fusion_agent")
else:
    ARTIFACTS_ROOT = Path("/content/artifacts/late_fusion_agent")
print("DATA_DIR:", DATA_DIR)
print("ARTIFACTS_ROOT:", ARTIFACTS_ROOT)


## 10. Mock data generation


In [ ]:
"""Generate synthetic BTCUSDT 1-min OHLCV (geometric Brownian motion).
Skipped automatically when USE_MOCK_DATA = False."""
import numpy as np
import pyarrow as pa
import pyarrow.parquet as pq


def _gen_month(n_minutes, start_price, seed):
    rng = np.random.default_rng(seed)
    sigma = 0.0008
    lr = rng.normal(0.0, sigma, size=n_minutes)
    closes = start_price * np.exp(np.cumsum(lr))
    opens = np.concatenate([[start_price], closes[:-1]])
    noise = np.abs(rng.normal(0, sigma, size=n_minutes)) * closes
    highs = np.maximum(opens, closes) + noise
    lows = np.minimum(opens, closes) - noise
    vols = rng.lognormal(3.0, 1.0, size=n_minutes).astype(np.float32)
    return {
        "open": opens.astype(np.float32),
        "high": highs.astype(np.float32),
        "low": lows.astype(np.float32),
        "close": closes.astype(np.float32),
        "volume": vols,
    }


if USE_MOCK_DATA:
    kd = DATA_DIR / "BTCUSDT" / "klines_1m"
    kd.mkdir(parents=True, exist_ok=True)
    start_price = 42000.0
    for i, m in enumerate(MOCK_MONTHS):
        out = kd / f"{m}.parquet"
        if out.exists():
            print("skip", out.name); continue
        d = _gen_month(MOCK_MINUTES_PER_MONTH, start_price, 42 + i)
        start_price = float(d["close"][-1])
        pq.write_table(pa.table(d), out)
        print(f"wrote {out.name}  rows={MOCK_MINUTES_PER_MONTH}")
    print("files:", sorted(p.name for p in kd.glob("*.parquet")))
else:
    print("USE_MOCK_DATA=False — expecting real data at", DATA_DIR)


## 11. make_split helper


In [ ]:
"""Helper: discover parquet files for a month range."""
from pathlib import Path


def make_split(data_dir: Path, symbol: str, start_month: str, end_month: str):
    kd = data_dir / symbol / "klines_1m"
    if not kd.exists():
        raise FileNotFoundError(kd)
    return sorted([f for f in kd.glob("*.parquet") if start_month <= f.stem <= end_month])


## 12. Main — train A, B, Meta; eval on test


In [ ]:
"""Main: build datasets, models, train A/B/Meta, evaluate."""
import random
import numpy as np
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report

random.seed(42); np.random.seed(42); torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

sc = CFG["sequence"]; tc = CFG["tokenizer"]
train_files = make_split(DATA_DIR, SYMBOL, *CFG["data"]["train_months"])
val_files   = make_split(DATA_DIR, SYMBOL, *CFG["data"]["val_months"])
test_files  = make_split(DATA_DIR, SYMBOL, *CFG["data"]["test_months"])
print(f"train={len(train_files)} val={len(val_files)} test={len(test_files)}")

def build_ds(files):
    return FusionDataset(files, seq_len=sc["length"],
        target_horizon=sc["target_horizon"], target_threshold=sc["target_threshold"],
        range_pct=tc["delta"]["range_pct"], step_pct=tc["delta"]["step_pct"],
        n_bins=tc["bucket"]["n_bins"])

train_ds = build_ds(train_files)
val_ds   = build_ds(val_files)
test_ds  = build_ds(test_files)
print(f"train_ds={len(train_ds)} val_ds={len(val_ds)} test_ds={len(test_ds)}")

bs = CFG["training"]["batch_size"]
train_dl = DataLoader(train_ds, batch_size=bs, shuffle=True,  collate_fn=collate, num_workers=0)
val_dl   = DataLoader(val_ds,   batch_size=bs, shuffle=False, collate_fn=collate, num_workers=0)
test_dl  = DataLoader(test_ds,  batch_size=bs, shuffle=False, collate_fn=collate, num_workers=0)

ma = CFG["model_a"]
model_a = PriceTransformer(
    delta_vocab_size=ma["delta_vocab_size"], bucket_vocab_size=ma["bucket_vocab_size"],
    delta_emb_dim=ma["delta_emb_dim"], bucket_emb_dim=ma["bucket_emb_dim"],
    hidden_dim=ma["hidden_dim"], num_layers=ma["num_layers"], num_heads=ma["num_heads"],
    ffn_dim=ma["ffn_dim"], dropout=ma["dropout"], num_classes=ma["num_classes"],
    seq_len=sc["length"])
mb = CFG["model_b"]
model_b = IndicatorModel(
    vocab_sizes=mb["vocab_sizes"], emb_dim=mb["emb_dim"], hidden_dim=mb["hidden_dim"],
    num_layers=mb["num_layers"], num_heads=mb["num_heads"], ffn_dim=mb["ffn_dim"],
    dropout=mb["dropout"], num_classes=mb["num_classes"], seq_len=sc["length"])
print(f"A params={sum(p.numel() for p in model_a.parameters()):,}  "
      f"B params={sum(p.numel() for p in model_b.parameters()):,}")

tr_cfg = CFG["training"]
trainer = FusionTrainer(model_a, model_b, train_dl, val_dl,
    epochs_a=tr_cfg["epochs_a"], epochs_b=tr_cfg["epochs_b"], epochs_meta=tr_cfg["epochs_meta"],
    lr=tr_cfg["learning_rate"], weight_decay=tr_cfg["weight_decay"],
    early_stop_patience=tr_cfg["early_stop_patience"], device=tr_cfg["device"],
    checkpoint_dir=Path(tr_cfg["checkpoint_dir"]))
meta = trainer.train_all()

print("\n=== Test evaluation (meta-model) ===")
dev = trainer.device
model_a.train(mode=False); model_b.train(mode=False); meta.train(mode=False)
all_preds, all_labels = [], []
with torch.no_grad():
    for b in test_dl:
        la = model_a(b[0].to(dev), b[1].to(dev), b[2].to(dev))
        lb = model_b([t.to(dev) for t in b[3]])
        logits = meta(la, lb)
        all_preds.extend(logits.argmax(-1).cpu().numpy().tolist())
        all_labels.extend(b[4].numpy().tolist())
print(classification_report(all_labels, all_preds,
    target_names=["DOWN","FLAT","UP"], zero_division=0))


## 13. Save artifacts to Google Drive
Persists all three checkpoints, fitted tokenizers, config and test metrics into `ARTIFACTS_ROOT` (Drive or `/content/artifacts`).


In [ ]:
"""Save checkpoints, tokenizers, metrics and config to ARTIFACTS_ROOT."""
import json
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = ARTIFACTS_ROOT / f"run_{RUN_TAG}"
(RUN_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
(RUN_DIR / "tokenizers").mkdir(parents=True, exist_ok=True)

ckpt_dir = Path(CFG["training"]["checkpoint_dir"])
for name in ["model_a_best.pt", "model_b_best.pt", "meta_model.pt"]:
    src = ckpt_dir / name
    if src.exists():
        shutil.copy(src, RUN_DIR / "checkpoints" / name)

np.save(RUN_DIR / "tokenizers" / "vol_boundaries.npy", train_ds.vt.boundaries)
np.save(RUN_DIR / "tokenizers" / "vb_boundaries.npy",  train_ds.bt.boundaries)
with open(RUN_DIR / "tokenizers" / "delta_params.json", "w") as f:
    json.dump({
        "range_pct": train_ds.dt.range_pct,
        "step_pct":  train_ds.dt.step_pct,
    }, f, indent=2)
train_ds.it.save(RUN_DIR / "tokenizers" / "indicators")

with open(RUN_DIR / "config.json", "w") as f:
    json.dump(CFG, f, indent=2, default=str)

test_report_dict = classification_report(
    all_labels, all_preds, target_names=["DOWN","FLAT","UP"],
    zero_division=0, output_dict=True)
with open(RUN_DIR / "test_metrics.json", "w") as f:
    json.dump({
        "report":           test_report_dict,
        "confusion_matrix": confusion_matrix(all_labels, all_preds).tolist(),
    }, f, indent=2)

np.savez_compressed(
    RUN_DIR / "predictions.npz",
    preds=np.asarray(all_preds, dtype=np.int8),
    labels=np.asarray(all_labels, dtype=np.int8),
)

print(f"saved to: {RUN_DIR}")
for p in sorted(RUN_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(RUN_DIR)!s:40s}  {p.stat().st_size:>10,} B")
